# Mini-Workshop 2: Selection Bias in Tuning Curves

An orientation-tuning experiment was run in two conditions (control and test).
For 200 V1 neurons, spike counts were recorded at 12 directions (30-degree
spacing, 10 repeats each). A colleague ran the standard analysis below and
concluded that the test condition reduces neural responsiveness.

Run the cells to reproduce the analysis, then work through the exercise to
evaluate whether the conclusion holds up.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import wilcoxon

plt.rcParams['font.family']       = 'sans-serif'
plt.rcParams['font.sans-serif']   = ['DejaVu Sans', 'Arial']
plt.rcParams['axes.spines.top']   = False
plt.rcParams['axes.spines.right'] = False
plt.rcParams['pdf.fonttype']      = 42
plt.rcParams['ps.fonttype']       = 42
CONTROL, TEST = 0, 1
COND_COLOR    = {'control': 'k', 'test': '0.6'}
COND_LABEL    = {'control': 'Control', 'test': 'Test'}
rng = np.random.default_rng(0)

def load(fname):
    """Load spike count dataset. Returns counts, duration, orientations, condition names."""
    d = np.load(fname, allow_pickle=True)
    return (d['counts'], float(d['duration']), d['orientations'],
            [str(x) for x in d['condition_names']])

def preferred_ori(rate_by_ori):
    """Find each neuron's preferred direction, breaking ties at random.
    Counts are small integers so exact ties are common; np.argmax's first-index
    rule would bias the preferred direction toward 0 deg."""
    return (rate_by_ori + rng.uniform(0, 1e-9, rate_by_ori.shape)).argmax(1)

def roll_to_center(rate_by_ori, pref_idx):
    """Circularly shift each neuron's tuning curve so its preferred direction
    sits at the center column, enabling averaging across neurons."""
    n_ori  = rate_by_ori.shape[1]
    center = n_ori // 2
    return np.array([np.roll(rate_by_ori[i], center - pref_idx[i])
                     for i in range(rate_by_ori.shape[0])])


In [ ]:
counts, duration, orientations, condition_names = load('data/tuned.npz')
n_neurons, n_ori, n_cond, n_trials = counts.shape
print(f'{n_neurons} neurons x {n_ori} directions x {n_cond} conditions x {n_trials} trials')


<div style="border-left: 3px solid #000; padding: 1px; padding-left: 10px; background: #F0FAFF; max-width: 90%; overflow-x: auto; color: #000000;">

## The analysis: preferred direction and tuning curves

The standard approach for comparing conditions at the preferred orientation:

1. Compute each neuron's mean firing rate across trials for each direction and condition.
2. Define each neuron's **preferred direction** as the direction with the highest
   mean response in the **control** condition.
3. Align both conditions to each neuron's preferred direction (shift so the preferred
   direction sits at the center) and average across neurons.

</div>


In [ ]:
rate      = counts / duration                    # spike counts -> firing rate (spikes/s)
mean_rate = rate.mean(axis=3)                    # average over trials -> (neuron, ori, condition)

# Define preferred direction from the CONTROL condition, then align both conditions to it.
pref_idx        = preferred_ori(mean_rate[:, :, CONTROL])
aligned_control = roll_to_center(mean_rate[:, :, CONTROL], pref_idx)
aligned_test    = roll_to_center(mean_rate[:, :, TEST],    pref_idx)

center  = n_ori // 2
rel_ori = (np.arange(n_ori) - center) * (orientations[1] - orientations[0])


In [ ]:
fig, ax = plt.subplots(figsize=(5.5, 4))
for cond, curves in [('control', aligned_control), ('test', aligned_test)]:
    m   = curves.mean(0)
    sem = curves.std(0) / np.sqrt(n_neurons)
    ax.errorbar(rel_ori, m, yerr=sem, color=COND_COLOR[cond], lw=2,
                marker='o', ms=4, capsize=2, label=COND_LABEL[cond])
ax.set_xlabel('direction relative to preferred (deg)')
ax.set_ylabel('evoked firing rate (spikes/s)')
ax.set_title('Mean tuning curve (n = %d)' % n_neurons)
ax.legend(frameon=False)
plt.show()

# Test at the preferred direction (center column), neuron by neuron.
p_ctrl = aligned_control[:, center]
p_test = aligned_test[:, center]
w      = wilcoxon(p_ctrl, p_test)
print(f'Firing rate at preferred direction:')
print(f'  Control : {p_ctrl.mean():.2f} +/- {p_ctrl.std()/np.sqrt(n_neurons):.2f} sp/s')
print(f'  Test    : {p_test.mean():.2f} +/- {p_test.std()/np.sqrt(n_neurons):.2f} sp/s')
print(f'  Wilcoxon signed-rank test: p = {w.pvalue:.2e}')


<div style="border-left: 3px solid #000; padding: 1px; padding-left: 10px; background: #F0FAFF; max-width: 90%; overflow-x: auto; color: #000000;">

The analysis shows clear tuning with a large, highly significant drop at the
preferred direction in the test condition (p < 1e-5). A colleague concludes:

> *"These neurons are orientation tuned, and the test manipulation reduces
> responsiveness: at each neuron's preferred orientation, the evoked firing
> rate is markedly higher in control than test."*

**The key conclusion (the reduction) is not supported by this analysis.** Work
through the exercise below to find out why.

</div>


<div style="border-left: 3px solid #07bc0a; padding: 1px; padding-left: 10px; background: #DFF0D8; max-width: 90%; overflow-x: auto; color: #000000;">

### Exercise: Evaluate the tuning curve analysis

<ol>

<li><strong>Identify the circular step.</strong> The analysis uses the control data
twice: once to define each neuron's preferred direction, and once to measure
the firing rate at that direction. What problem does this cause? What would
happen to noisy estimates if you always selected the direction with the
highest value and then measured it again?
<details>
<summary>Hint</summary>

*This is the <strong>winner's curse</strong> (or regression to the mean). Selecting
the maximum of 12 noisy numbers inflates the estimate upward. A second
independent measurement at that same location will, on average, be lower
- not because the response changed, but because noise drove the first
estimate up.*

</details>
</li>

<br>

<li><strong>Predict the outcome if you select from the other condition.</strong> If the
reduction were real, it should not matter which condition defines the preferred
direction. Predict: what would the figure look like if you used the
<em>test</em> condition to define preferred direction instead of control?
<details>
<summary>Hint</summary>

*The inflation lands on whichever condition you selected with. If you select
from test, test gets inflated - and now test > control at the preferred
direction.*

</details>
</li>

<br>

<li><strong>Design a fair test.</strong> How would you separately answer (a) are these
neurons genuinely tuned, and (b) is the condition difference real? What data
would each analysis use to define the preferred direction, and what data
would it use to measure?
<details>
<summary>Hint</summary>

*Use <strong>cross-validation</strong>: split the control trials in half, define the
preferred direction on one half, and read out the tuning curve on the other
half (and on the test trials, which were never used to pick the direction).
Average over many random splits.*

</details>
</li>

</ol>

</div>


## Solution - Act 1: Real tuning, fake condition difference


In [ ]:
# Cross-validation: for many random splits of control trials, define the
# preferred direction on one half and read out on the other half (and test).
# This breaks the circular step: the preferred direction is never defined on
# the data being measured.
n_splits = 60
half     = n_trials // 2
accC = np.zeros((n_neurons, n_ori))
accT = np.zeros((n_neurons, n_ori))

for _ in range(n_splits):
    perm = rng.permutation(n_trials)
    pick = rate[:, :, CONTROL, perm[:half]].mean(2)   # choose preferred direction here...
    read = rate[:, :, CONTROL, perm[half:]].mean(2)   # ...read control on independent data
    pk   = preferred_ori(pick)
    accC += roll_to_center(read, pk)
    accT += roll_to_center(mean_rate[:, :, TEST], pk)  # test was never used to pick

cvC = accC / n_splits    # mean held-out control tuning curve
cvT = accT / n_splits    # mean held-out test tuning curve

fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
for ax, title, cc, ct in [
        (axes[0], 'Naive (select and read from same control data)',
         aligned_control, aligned_test),
        (axes[1], 'Cross-validated (independent selection)',
         cvC, cvT)]:
    ax.errorbar(rel_ori, cc.mean(0), cc.std(0)/np.sqrt(n_neurons),
                color='k', lw=2, marker='o', ms=4, label='Control')
    ax.errorbar(rel_ori, ct.mean(0), ct.std(0)/np.sqrt(n_neurons),
                color='0.6', lw=2, marker='o', ms=4, label='Test')
    ax.set_xlabel('direction relative to preferred (deg)')
    ax.set_title(title, fontsize=10); ax.legend(frameon=False)
axes[0].set_ylabel('firing rate (spikes/s)')
plt.tight_layout(); plt.show()

print(f'Naive gap at preferred direction:          {aligned_control[:,center].mean()-aligned_test[:,center].mean():.2f} sp/s')
print(f'Cross-validated gap at preferred direction: {cvC[:,center].mean()-cvT[:,center].mean():.2f} sp/s')
print(f'Cross-validated tuning: peak={cvC[:,center].mean():.2f}  flank={np.delete(cvC,center,1).mean():.2f} sp/s (still tuned)')


In [ ]:
# The bias follows whichever condition you select on.
# If the reduction were real, swapping to select from TEST should not reverse it.
pref_test  = preferred_ori(mean_rate[:, :, TEST])
sw_control = roll_to_center(mean_rate[:, :, CONTROL], pref_test)
sw_test    = roll_to_center(mean_rate[:, :, TEST],    pref_test)

fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
for ax, title, cc, ct in [
        (axes[0], 'Preferred direction from CONTROL', aligned_control, aligned_test),
        (axes[1], 'Preferred direction from TEST',    sw_control,      sw_test)]:
    ax.errorbar(rel_ori, cc.mean(0), cc.std(0)/np.sqrt(n_neurons),
                color='k', lw=2, marker='o', ms=4, label='Control')
    ax.errorbar(rel_ori, ct.mean(0), ct.std(0)/np.sqrt(n_neurons),
                color='0.6', lw=2, marker='o', ms=4, label='Test')
    ax.set_xlabel('direction relative to preferred (deg)')
    ax.set_title(title, fontsize=10); ax.legend(frameon=False)
axes[0].set_ylabel('firing rate (spikes/s)')
plt.tight_layout(); plt.show()
print('When preferred direction is selected from TEST, the gap reverses:')
print(f'  test@pref={sw_test[:,center].mean():.2f}  control@pref={sw_control[:,center].mean():.2f} sp/s')


## Solution - Act 2: No tuning at all, fake everything


In [ ]:
# Load the untuned population - no orientation tuning whatsoever.
# The same pipeline produces a sharp tuning curve and a large condition
# difference entirely from noise.
counts, duration, orientations, condition_names = load('data/untuned.npz')
n_neurons, n_ori, n_cond, n_trials = counts.shape
rate       = counts / duration
mean_rate  = rate.mean(axis=3)
center     = n_ori // 2
rel_ori    = (np.arange(n_ori) - center) * (orientations[1] - orientations[0])

pref_idx        = preferred_ori(mean_rate[:, :, CONTROL])
aligned_control = roll_to_center(mean_rate[:, :, CONTROL], pref_idx)
aligned_test    = roll_to_center(mean_rate[:, :, TEST],    pref_idx)
print(f'naive (untuned): control@pref={aligned_control[:,center].mean():.2f}  '
      f'test@pref={aligned_test[:,center].mean():.2f}  '
      f'p={wilcoxon(aligned_control[:,center], aligned_test[:,center]).pvalue:.1e}')


In [ ]:
# Cross-validate the untuned data: the entire peak vanishes,
# not just the condition difference.
n_splits = 60; half = n_trials // 2
accC = np.zeros((n_neurons, n_ori))
for _ in range(n_splits):
    perm = rng.permutation(n_trials)
    pick = rate[:, :, CONTROL, perm[:half]].mean(2)
    read = rate[:, :, CONTROL, perm[half:]].mean(2)
    accC += roll_to_center(read, preferred_ori(pick))
cvC = accC / n_splits

fig, ax = plt.subplots(figsize=(5.5, 4))
ax.errorbar(rel_ori, aligned_control.mean(0),
            aligned_control.std(0)/np.sqrt(n_neurons),
            color='k', lw=2, marker='o', ms=4, label='naive (biased)')
ax.errorbar(rel_ori, cvC.mean(0), cvC.std(0)/np.sqrt(n_neurons),
            color='tab:green', lw=2, marker='D', ms=4, label='cross-validated')
ax.set_xlabel('direction relative to preferred (deg)')
ax.set_ylabel('firing rate (spikes/s)')
ax.set_title('No tuning: cross-validation flattens everything')
ax.legend(frameon=False)
plt.show()
print(f'naive peak={aligned_control[:,center].mean():.2f}  '
      f'cross-validated peak={cvC[:,center].mean():.2f} sp/s')


<div style="border-left: 3px solid #000; padding: 1px; padding-left: 10px; background: #F0FAFF; max-width: 90%; overflow-x: auto; color: #000000;">

## Key takeaway

Both datasets were generated with control and test conditions identical - there
is no condition difference anywhere. The apparent reduction is entirely a
selection artifact.

- **Never select and measure on the same data (circular analysis / double dipping).**
  Choosing the preferred direction by argmax and then reporting the response at
  that direction biases the selected value upward. Any statistic used to select
  must be computed on data independent of the statistic you report.
- **Selection bias can inflate a comparison or manufacture the whole effect.**
  With real tuning it invents a condition difference (Act 1); with no tuning it
  invents the tuning curve itself (Act 2).
- **The bias follows whichever condition you select on.** If the reduction were
  real it would not flip when you swap which condition defines the preferred direction.
- **Cross-validation separates the two questions.** It correctly shows that the
  tuning is real (the peak survives independent data) while the condition
  difference is not (it vanishes with independent selection).

</div>
